In [60]:
import numpy as np

In [61]:
datos_experimentales = np.load('conversion_datos_Exp.npz')

alpha = datos_experimentales['alpha']
delta = datos_experimentales['delta']
tiempos_dias_jd = datos_experimentales['tiempos_dias_jd']



In [62]:
alpha_1 = alpha[0:4]
delta_1 = delta[0:4]

tiempos_uso_jd_1 = tiempos_dias_jd[0:4]  
tiempos_uso_jd_1

array([2461081.49722222, 2461100.50972222, 2461102.47638889,
       2461111.50138889])

In [63]:
tiempo_central_1 = np.mean(tiempos_uso_jd_1)

In [64]:
#Construcción del vector rho

rho = np.zeros((4, 3))

for i in range(4):
    rho[i, 0] = np.cos(delta_1[i]) * np.cos(alpha_1[i])  # x
    rho[i, 1] = np.cos(delta_1[i]) * np.sin(alpha_1[i])  # y
    rho[i, 2] = np.sin(delta_1[i])                     # z

In [65]:
#Interpolación de los componentes de rho (lambda, mu y nu)

#Como los dias julianos son muy grandes vamos a reescalar para ver si funciona bien c: Revisar

tiempo_relativo = tiempos_uso_jd_1 - tiempos_uso_jd_1[0]

poly_lambda = np.polyfit(tiempo_relativo, rho[:, 0], 3)
poly_mu = np.polyfit(tiempo_relativo, rho[:, 1], 3)
poly_nu = np.polyfit(tiempo_relativo, rho[:, 2], 3)

tiempo_central_reescalado = tiempo_central_1 - tiempos_uso_jd_1[0] 

In [66]:
#Para lambda

l_val = np.polyval(poly_lambda, tiempo_central_reescalado)

derivada_1_l = np.polyder(poly_lambda,1)
l_dval = np.polyval(derivada_1_l, tiempo_central_reescalado)

derivada_2_l = np.polyder(poly_lambda,2)
l_ddval = np.polyval(derivada_2_l, tiempo_central_reescalado)

#Para mu

mu_val = np.polyval(poly_mu, tiempo_central_reescalado)

derivada_1_mu = np.polyder(poly_mu,1)
mu_dval = np.polyval(derivada_1_mu, tiempo_central_reescalado)

derivada_2_mu = np.polyder(poly_mu,2)
mu_ddval = np.polyval(derivada_2_mu, tiempo_central_reescalado)

#Para nu

nu_val = np.polyval(poly_nu, tiempo_central_reescalado)

derivada_1_nu = np.polyder(poly_nu,1)
nu_dval = np.polyval(derivada_1_nu, tiempo_central_reescalado)

derivada_2_nu = np.polyder(poly_nu,2)
nu_ddval = np.polyval(derivada_2_nu, tiempo_central_reescalado)

In [67]:
#l_val, mu_val y nu_val están dentro del rango de -1 a 1.
#l_dval está en el rango de 10e-3, y mu_dval y nu_dval están en el rango de 10e-4.
#l_ddval, mu_ddval y nu_ddval están en el rango de 10e-5. Esto es consistente con lo que se espera para las posiciones y velocidades de un planeta en el sistema solar.

In [68]:
#Datos de las efemérides del Sol

X = -0.92166819869956
Y = 0.3624168676580381
Z = -1.877615350367358e-5


#Distancia desde el Sol hasta el planeta Tierra (aproximadamente 1UA)
R = np.sqrt(X**2 + Y**2 + Z**2)

In [69]:
#Cálculo de las matrices D(rho) y D1(rho y R)

matriz_D = np.array([
    [l_val,  l_dval,  l_ddval],
    [mu_val, mu_dval, mu_ddval],
    [nu_val, nu_dval, nu_ddval]
])

# Calcular el determinante D
D = np.linalg.det(matriz_D)
D

#D está dando 4.5923456027e-08. O los datos están muys eguidos entre sí, o estamos haciendo algo mal. Revisar.

#k es la constante de gravitación universal en unidades de UA^3/dia^{2}

k = 0.01720209895
k2 = k**2

matriz_D1 = np.array([
    [l_val, l_dval, X],
    [mu_val, mu_dval, Y],
    [nu_val, nu_dval, Z]
])

#Calcular el determiante D1
D1 = k2* np.linalg.det(matriz_D1)
D1

np.float64(1.3002171451210513e-07)

In [70]:
#Intento preliminar
intento = D1/D * (1/(0.99)**3 - 1/(5.2)**3)
intento

np.float64(2.8977999445141935)

In [71]:
cos_phi = (l_val*X + mu_val*Y +nu_val*Z)/(R)

# Bucle para hallar r y rho

r_obj = 5.2  # Distancia promedio Sol Y Júpiter en UA
rho_old = 0
tolerancia = 1e-8
max_iter = 50
i = 0

print(f"{'Iter':<5} | {'rho (UA)':<15} | {'r (UA)':<15}")
print("-" * 40)

# Bucle
while i < max_iter:
    rho_new = (D1 / D) * (1/R**3 - 1/r_obj**3)
    
    # Ley de los Cosenos: Hallamos el nuevo r a partir de rho
    # "El signo es positivo porque estamos asumiendo que R es desde el Sol hasta la Tierra"
    r_obj = np.sqrt(rho_new**2 + R**2 + 2 * rho_new * R * cos_phi)
    
    print(f"{i:<5} | {rho_new:<15.8f} | {r_obj:<15.8f}")
    
    # Para cuando rho ya no cambia
    if abs(rho_new - rho_old) < tolerancia:
        print("-" * 40)
        print(f"¡Convergencia lograda en {i} iteraciones!")
        break
        
    rho_old = rho_new
    i += 1

rho_final = rho_new
r_final = r_obj

print(f"\nDistancia Tierra-Júpiter (rho): {rho_final:.6f} UA")
print(f"Distancia Sol-Júpiter (r): {r_final:.6f} UA")

Iter  | rho (UA)        | r (UA)         
----------------------------------------
0     | 2.89459532      | 3.55457597     
1     | 2.85169098      | 3.51282214     
2     | 2.84941624      | 3.51060918     
3     | 2.84929265      | 3.51048894     
4     | 2.84928592      | 3.51048240     
5     | 2.84928556      | 3.51048204     
6     | 2.84928554      | 3.51048202     
7     | 2.84928554      | 3.51048202     
----------------------------------------
¡Convergencia lograda en 7 iteraciones!

Distancia Tierra-Júpiter (rho): 2.849286 UA
Distancia Sol-Júpiter (r): 3.510482 UA


In [72]:
#Finalmente, el vectoor de jupiter es

x_jup = X + rho_final * l_val
y_jup = Y + rho_final * mu_val
z_jup = Z + rho_final * nu_val

vector_r_jupiter = np.array([x_jup, y_jup, z_jup])
vector_r_jupiter

array([-1.68348241,  2.87333691,  1.11078184])

In [73]:
np.linalg.norm(vector_r_jupiter)

np.float64(3.5105575514767637)

Con la fecha 2 hasta la 5

In [74]:
tiempos_uso_jd_2 = tiempos_dias_jd[1:5]  
tiempos_uso_jd_2

array([2461100.50972222, 2461102.47638889, 2461111.50138889,
       2461119.54305556])

In [75]:
tiempo_central_2 = np.mean(tiempos_uso_jd_2)
tiempo_central_2

np.float64(2461108.507638889)

In [76]:
#Construcción del vector rho

alpha_2 = alpha[1:5]
delta_2 = delta[1:5]

rho_2 = np.zeros((4, 3))

for i in range(4):
    rho_2[i, 0] = np.cos(delta_2[i]) * np.cos(alpha_2[i])  # x
    rho_2[i, 1] = np.cos(delta_2[i]) * np.sin(alpha_2[i])  # y
    rho_2[i, 2] = np.sin(delta_2[i])                     # z

In [77]:
#Interpolación de los componentes de rho (lambda, mu y nu)

#Como los dias julianos son muy grandes vamos a reescalar para ver si funciona bien c: Revisar

tiempo_relativo_2 = tiempos_uso_jd_2 - tiempos_uso_jd_2[0]

poly_lambda_2 = np.polyfit(tiempo_relativo_2, rho_2[:, 0], 3)
poly_mu_2 = np.polyfit(tiempo_relativo_2, rho_2[:, 1], 3)
poly_nu_2 = np.polyfit(tiempo_relativo_2, rho_2[:, 2], 3)

tiempo_central_reescalado_2 = tiempo_central_2 - tiempos_uso_jd_2[0] 

In [78]:
#Para lambda

l_val_2 = np.polyval(poly_lambda_2, tiempo_central_reescalado_2)

derivada_1_l_2 = np.polyder(poly_lambda_2,1)
l_dval_2 = np.polyval(derivada_1_l_2, tiempo_central_reescalado_2)

derivada_2_l_2 = np.polyder(poly_lambda_2,2)
l_ddval_2 = np.polyval(derivada_2_l_2, tiempo_central_reescalado_2)

#Para mu

mu_val_2 = np.polyval(poly_mu_2, tiempo_central_reescalado_2)

derivada_1_mu_2 = np.polyder(poly_mu_2,1)
mu_dval_2 = np.polyval(derivada_1_mu_2, tiempo_central_reescalado_2)

derivada_2_mu_2 = np.polyder(poly_mu_2,2)
mu_ddval_2 = np.polyval(derivada_2_mu_2, tiempo_central_reescalado_2)

#Para nu

nu_val_2 = np.polyval(poly_nu_2, tiempo_central_reescalado_2)

derivada_1_nu_2 = np.polyder(poly_nu_2,1)
nu_dval_2 = np.polyval(derivada_1_nu_2, tiempo_central_reescalado_2)

derivada_2_nu_2 = np.polyder(poly_nu_2,2)
nu_ddval_2 = np.polyval(derivada_2_nu_2, tiempo_central_reescalado_2)

In [79]:
X_2 = -0.9712751310029790
Y_2 = 0.2052987725362495
Z_2 = -4.038939877607600e-06

R_2 = np.sqrt(X_2**2 + Y_2**2 + Z_2**2)
R_2


np.float64(0.992735093630752)

In [80]:
#Cálculo de las matrices D(rho) y D1(rho y R)

matriz_D_2 = np.array([
    [l_val_2,  l_dval_2,  l_ddval_2],
    [mu_val_2, mu_dval_2, mu_ddval_2],
    [nu_val_2, nu_dval_2, nu_ddval_2]
])

# Calcular el determinante D
D_2 = np.linalg.det(matriz_D_2)
print(D_2)

#D está dando 6.040795640314e-08. O los datos están muys eguidos entre sí, o estamos haciendo algo mal. Revisar.

#k es la constante de gravitación universal en unidades de UA^3/dia^{2}

k = 0.01720209895
k2 = k**2

matriz_D1_2 = np.array([
    [l_val_2, l_dval_2, X_2],
    [mu_val_2, mu_dval_2, Y_2],
    [nu_val_2, nu_dval_2, Z_2]
])

#Calcular el determiante D1
D1_2 = k2* np.linalg.det(matriz_D1_2)
print(D1_2)

6.040795640314286e-08
4.381487600765345e-08


In [81]:
#Intento preliminar
intento_2 = D1_2/D_2 * (1/(0.99)**3 - 1/(5.2)**3)
intento_2

np.float64(0.7423599236935724)

In [82]:
cos_phi_2 = (l_val_2*X_2 + mu_val_2*Y_2 +nu_val_2*Z_2)/(R_2)

# Bucle para hallar r y rho

r_obj_2 = 5.2  # Distancia promedio Sol Y Júpiter en UA
rho_old_2 = 0
tolerancia = 1e-8
max_iter = 50
i = 0

print(f"{'Iter':<5} | {'rho (UA)':<15} | {'r (UA)':<15}")
print("-" * 40)

# Bucle
while i < max_iter:
    rho_new_2 = (D1_2 / D_2) * (1/R_2**3 - 1/r_obj_2**3)
    
    # Ley de los Cosenos: Hallamos el nuevo r a partir de rho
    # "El signo es positivo porque estamos asumiendo que R es desde el Sol hasta la Tierra"
    r_obj_2 = np.sqrt(rho_new_2**2 + R_2**2 + 2 * rho_new_2 * R_2 * cos_phi_2)
    
    print(f"{i:<5} | {rho_new_2:<15.8f} | {r_obj_2:<15.8f}")
    
    # Para cuando rho ya no cambia
    if abs(rho_new_2 - rho_old_2) < tolerancia:
        print("-" * 40)
        print(f"¡Convergencia lograda en {i} iteraciones!")
        break
        
    rho_old_2 = rho_new_2
    i += 1

rho_final_2 = rho_new_2
r_final_2 = r_obj_2

print(f"\nDistancia Tierra-Júpiter (rho): {rho_final_2:.6f} UA")
print(f"Distancia Sol-Júpiter (r): {r_final_2:.6f} UA")

Iter  | rho (UA)        | r (UA)         
----------------------------------------
0     | 0.73619845      | 1.46789467     
1     | 0.51203644      | 1.29766456     
2     | 0.40943176      | 1.22554983     
3     | 0.34732212      | 1.18408463     
4     | 0.30445973      | 1.15654784     
5     | 0.27250394      | 1.13663577     
6     | 0.24742906      | 1.12140139     
7     | 0.22702413      | 1.10926820     
8     | 0.20996152      | 1.09931040     
9     | 0.19538982      | 1.09094562     
10    | 0.18273475      | 1.08378765     
11    | 0.17159308      | 1.07756936     
12    | 0.16167231      | 1.07209957     
13    | 0.15275441      | 1.06723749     
14    | 0.14467311      | 1.06287695     
15    | 0.13729910      | 1.05893620     
16    | 0.13053010      | 1.05535117     
17    | 0.12428400      | 1.05207088     
18    | 0.11849400      | 1.04905417     
19    | 0.11310514      | 1.04626744     
20    | 0.10807172      | 1.04368294     
21    | 0.10335539      | 1.0412775

In [83]:
#Finalmente, el vectoor de jupiter es

x_jup_2 = X_2 + rho_final_2 * l_val_2
y_jup_2 = Y_2 + rho_final_2 * mu_val_2
z_jup_2 = Z_2 + rho_final_2 * nu_val_2

vector_r_jupiter_2 = np.array([x_jup_2, y_jup_2, z_jup_2])
vector_r_jupiter_2

array([-0.97988186,  0.23560421,  0.0133841 ])

In [84]:
np.linalg.norm(vector_r_jupiter_2)

np.float64(1.0078972888173245)